- Knowledge Graph is  a database that stores information in nodes and relationships.
- both nodes and relationships can have properties
- nodes can be given labels to group them together
- 

In [3]:
cypher="""match (n)
return count(n)"""

In [ ]:
result=kg.query(cypher)
result

In [ ]:
cypher="""match (n)
return count(n) AS numberOfNodes"""
result=kg.query(cypher)

In [1]:
pip install langchain-neo4j neo4j python-dotenv langchain-community

  Using cached langchain_neo4j-0.8.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached neo4j-6.1.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached neo4j_graphrag-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached json_repair-0.44.1-py3-none-any.whl.metadata (12 kB)
  Using cached pypdf-6.6.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached types_pyyaml-6.0.12.20250915-py3-none-any.whl.metadata (1.7 kB)
Using cached langchain_neo4j-0.8.0-py3-none-any.whl (48 kB)
Using cached neo4j-6.1.0-py3-none-any.whl (325 kB)
Using cached neo4j_graphrag-1.12.0-py3-none-any.whl (207 kB)
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   --------- ------------------------------ 3.1/12.9 MB 15.4 MB/s eta 0:00:01
   ------------------ --------------------- 6.0/12.9 MB 14.7 MB/s eta 0:00:01
   -------------------------- ------------- 8.7/12.9 MB 14.1 MB/s eta 0:00:01
   ---------------------------------- ----- 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
embedchain 0.1.127 requires chromadb<0.6.0,>=0.5.10, but you have chromadb 1.0.7 which is incompatible.
embedchain 0.1.127 requires langchain<0.4.0,>=0.3.1, but you have langchain 1.0.8 which is incompatible.
embedchain 0.1.127 requires langchain-community<0.4.0,>=0.3.1, but you have langchain-community 0.4.1 which is incompatible.
embedchain 0.1.127 requires langchain-openai<0.3.0,>=0.2.1, but you have langchain-openai 1.0.3 which is incompatible.
embedchain 0.1.127 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.3.45 which is incompatible.
embedchain 0.1.127 requires pypdf<6.0.0,>=5.0.0, but you have pypdf 6.6.0 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
instructor 1.7.4 requires jiter<0.9,>=0.6.1, but you have jiter 0.12.0 which i

In [3]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph


In [5]:
NEO4J_URI= "bolt://localhost:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD=""

In [19]:
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="demo"
)

# Optional: Verify the connection and refresh the schema
kg.refresh_schema()
print("Connection established and schema refreshed.")

Connection established and schema refreshed.


In [25]:
query1 = """
        CREATE (p1:Person {name: 'Alice Johnson', age: 28, city: 'New York'})
        CREATE (p2:Person {name: 'Bob Smith', age: 35, city: 'San Francisco'})
        CREATE (p3:Person {name: 'Carol White', age: 42, city: 'Boston'})
        CREATE (p4:Person {name: 'David Brown', age: 31, city: 'Seattle'})
        RETURN 'Created 4 Person nodes' AS result
        """

In [27]:
query2 = """CREATE (c1:Company:TechCompany {
            name: 'TechCorp', 
            founded: 2010, 
            industry: 'Software',
            employees: 500
        })
        CREATE (c2:Company:TechCompany {
            name: 'DataSolutions', 
            founded: 2015, 
            industry: 'Data Analytics',
            employees: 150
        })
        RETURN 'Created 2 Company nodes with multiple labels' AS result
        """
        

In [29]:
query3 = """
        MATCH (p:Person {name: 'Alice Johnson'}), (c:Company {name: 'TechCorp'})
        CREATE (p)-[r:WORKS_AT {since: 2020, position: 'Software Engineer'}]->(c)
        RETURN p.name + ' works at ' + c.name AS result
        """
####(Alice Johnson) -[:WORKS_AT {since: 2020, position: 'Software Engineer'}]-> (TechCorp)

In [ ]:
query4 = """
        MATCH (p1:Person {name: 'Bob Smith'}), (c:Company {name: 'DataSolutions'})
        CREATE (p1)-[:WORKS_AT {since: 2019, position: 'Data Scientist'}]->(c)
        
        WITH p1
        MATCH (p2:Person {name: 'Alice Johnson'})
        CREATE (p1)-[:KNOWS {since: 2018, relationship: 'Friends'}]->(p2)
        
        RETURN 'Created multiple relationships' AS result
        """

####(Bob Smith) -[:WORKS_AT {since:2019, position:'Data Scientist'}]-> (DataSolutions)
###(Bob Smith) -[:KNOWS {since:2018, relationship:'Friends'}]-> (Alice Johnson)

In [ ]:
query5 = """
        MERGE (p:Person {name: 'Alice Johnson'})
        ON CREATE SET p.created = timestamp()
        ON MATCH SET p.lastSeen = timestamp()
        RETURN p.name, p.created, p.lastSeen
        """

In [ ]:
#Selecting All Nodes of a Type
       # Basic SELECT
query = """
        MATCH (p:Person)
        RETURN p.name AS name, p.age AS age, p.city AS city
        ORDER BY p.age
        """

In [ ]:
# Simple WHERE clause
query = """
        MATCH (p:Person)
        WHERE p.age > 30
        RETURN p.name AS name, p.age AS age
        """

In [ ]:
# Multiple conditions
query = """
        MATCH (p:Person)
        WHERE p.age >= 30 AND p.city IN ['New York', 'Boston']
        RETURN p.name AS name, p.age AS age, p.city AS city
        """

In [ ]:
# Query with relationship filtering
query = """
        MATCH (p1:Person)-[k:KNOWS]->(p2:Person)
        WHERE k.since < 2020
        RETURN p1.name AS person1, p2.name AS person2, k.since AS friendsSince
        """

In [ ]:
#aggregation
query = """
        MATCH (p:Person)
        RETURN COUNT(p) AS totalPeople,
               AVG(p.age) AS avgAge,
               MIN(p.age) AS youngest,
               MAX(p.age) AS oldest
        """

In [ ]:
# Group by
query = """
        MATCH (p:Person)
        RETURN p.city AS city, 
               COUNT(p) AS peopleCount,
               AVG(p.age) AS avgAge
        ORDER BY peopleCount DESC
        """

In [ ]:
# Update single property
query = """
        MATCH (p:Person {name: 'Alice Johnson'})
        SET p.age = 29, p.email = 'alice@example.com'
        RETURN p.name AS name, p.age AS age, p.email AS email
        """

In [ ]:
 ######Querying Relationships
        ########Find nodes connected by relationships
        """query = """
        MATCH (p:Person)-[r:WORKS_AT]->(c:Company)
        RETURN p.name AS employee, 
               r.position AS position, 
               c.name AS company,
               r.since AS startYear
        """

In [35]:
result =kg.query(query2)
print("\nQuery executed:")
print(query2)
print("\nResult:", result)


Query executed:
CREATE (c1:Company:TechCompany {
            name: 'TechCorp', 
            founded: 2010, 
            industry: 'Software',
            employees: 500
        })
        CREATE (c2:Company:TechCompany {
            name: 'DataSolutions', 
            founded: 2015, 
            industry: 'Data Analytics',
            employees: 150
        })
        RETURN 'Created 2 Company nodes with multiple labels' AS result
        

Result: [{'result': 'Created 2 Company nodes with multiple labels'}]
